In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import time
import threading

In [2]:
path = "C:\\Program Files\\MetaTrader 5\\terminal64.exe"
login = 5030595537
password = "-r7lRdVn"
server = "MetaQuotes-Demo"
timeout = 10000
portable = False
if mt5.initialize(path=path, login=login, password=password, server=server, timeout=timeout, portable=portable):
    print("Initialization successful")
else:
    print("Initialization not successful")

Initialization successful


In [3]:
symbol = "EURUSD"

timeframe = mt5.TIMEFRAME_H1
from datetime import datetime

import pytz

end_time = datetime.today().astimezone(pytz.utc)

eurusd_rates = mt5.copy_rates_from(symbol, timeframe, end_time, 10000)

In [7]:
data = pd.DataFrame(eurusd_rates)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   time         10000 non-null  int64  
 1   open         10000 non-null  float64
 2   high         10000 non-null  float64
 3   low          10000 non-null  float64
 4   close        10000 non-null  float64
 5   tick_volume  10000 non-null  uint64 
 6   spread       10000 non-null  int32  
 7   real_volume  10000 non-null  uint64 
dtypes: float64(4), int32(1), int64(1), uint64(2)
memory usage: 586.1 KB


In [ ]:
data.to_csv('EURUSD_10000.csv', index=False)

In [8]:
data['time'] = pd.to_datetime(data['time'], unit='s').dt.tz_localize('UTC')

In [10]:
data.tail()

,time,open,high,low,close,tick_volume,spread,real_volume
9995,2024-12-10 04:00:00+00:00,1.05529,1.05613,1.05515,1.05532,1673,4,0
9996,2024-12-10 05:00:00+00:00,1.05532,1.05576,1.05489,1.05564,1472,4,0
9997,2024-12-10 06:00:00+00:00,1.05564,1.05598,1.05560,1.05574,1129,4,0
9998,2024-12-10 07:00:00+00:00,1.05576,1.05625,1.05540,1.05619,1195,4,0
9999,2024-12-10 08:00:00+00:00,1.05618,1.05680,1.05608,1.05665,1354,4,0


In [6]:

rates = mt5.copy_rates_from(symbol, timeframe, 1000)
print(rates)
da = pd.DataFrame(rates)

None


In [15]:
da.head()

""


In [2]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import time
import threading
from datetime import datetime
import pytz

class SVMTradingBot:
    def __init__(self, account, password, server, model_path, symbol="EURUSD", timeframe=mt5.TIMEFRAME_H1):
        print("Initializing SVMTradingBot...")
        self.account = account
        self.password = password
        self.server = server
        self.model_path = model_path
        self.symbol = symbol
        self.timeframe = timeframe
        self.scaler = StandardScaler()
        self.model = joblib.load(model_path)
        self.running = False
        
        # Initialize and connect to MetaTrader 5
        print("Connecting to MetaTrader 5...")
        if not mt5.initialize():
            print("initialize() failed, error code =", mt5.last_error())
            quit()

        authorized = mt5.login(self.account, password=self.password, server=self.server)
        if not authorized:
            print("Failed to connect to account. Error:", mt5.last_error())
            quit()
        print("Connected to MetaTrader 5")

    def fetch_data(self):
        print("Fetching data...")
        end_time = datetime.today().astimezone(pytz.utc)
        rates = mt5.copy_rates_from(self.symbol, self.timeframe, end_time, 10000)
        
        # Check if rates data is fetched successfully
        if rates is None:
            print("Failed to fetch data. Error:", mt5.last_error())
            return pd.DataFrame()  # Return an empty DataFrame to avoid further errors

        data = pd.DataFrame(rates)
        print("Data fetched successfully. DataFrame shape:", data.shape)
        print(data.head())  # Print the first few rows of the DataFrame to check its structure
        
        data['time'] = pd.to_datetime(data['time'], unit='s')
        data.set_index('time', inplace=True)
        return data

    def calculate_features(self, data):
        print("Calculating features...")
        data['Returns'] = data['close'].pct_change()
        data['SMA_5'] = data['close'].rolling(window=5).mean()
        data['SMA_10'] = data['close'].rolling(window=10).mean()
        data['SMA_20'] = data['close'].rolling(window=20).mean()
        data['EMA_5'] = data['close'].ewm(span=5, adjust=False).mean()
        data['EMA_10'] = data['close'].ewm(span=10, adjust=False).mean()
        data['Momentum'] = data['close'] - data['close'].shift(5)
        data['Volatility'] = data['Returns'].rolling(window=5).std()
        data['ROC_5'] = data['close'].pct_change(periods=5)
        data['Price_Change'] = data['close'].pct_change()
        data['Lag_1'] = data['close'].shift(1)
        data['RSI'] = 100 - (100 / (1 + data['Returns'].rolling(window=14).mean() /
                                     abs(data['Returns'].rolling(window=14).mean())))
        data.replace([np.inf, -np.inf], np.nan, inplace=True)
        data.dropna(inplace=True)
        print("Features calculated successfully")
        return data

    def make_predictions(self, data):
        print("Making predictions...")
        X = data[['SMA_5', 'SMA_10', 'SMA_20', 'EMA_5', 'EMA_10', 'Momentum', 'Volatility', 'ROC_5', 'Price_Change', 'Lag_1', 'RSI']]
        X_scaled = self.scaler.fit_transform(X)
        y_pred = self.model.predict(X_scaled)
        data['Predicted_Signal'] = y_pred
        print("Predictions made successfully")
        return data

    def place_order(self, order_type, volume, price, stop_loss, take_profit):
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": self.symbol,
            "volume": volume,
            "type": order_type,
            "price": price,
            "sl": stop_loss,
            "tp": take_profit,
            "deviation": 10,
            "magic": 234000,
            "comment": "SVMTradingBot order",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_IOC,
        }
        result = mt5.order_send(request)
        return result

    def close_order(self, ticket):
        position = mt5.positions_get(ticket=ticket)
        if not position:
            print(f"Position with ticket {ticket} not found")
            return None

        order_type = mt5.ORDER_TYPE_SELL if position[0].type == mt5.ORDER_TYPE_BUY else mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(self.symbol).bid if order_type == mt5.ORDER_TYPE_BUY else mt5.symbol_info_tick(self.symbol).ask

        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": self.symbol,
            "volume": position[0].volume,
            "type": order_type,
            "price": price,
            "deviation": 10,
            "magic": 234000,
            "comment": "SVMTradingBot close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_IOC,
            "position": ticket
        }

        result = mt5.order_send(request)
        if result.retcode != mt5.TRADE_RETCODE_DONE:
            print(f"Close order failed, retcode={result.retcode}")
            result_dict = result._asdict()
            for field in result_dict.keys():
                print("   {}={}".format(field, result_dict[field]))
        else:
            print(f"Order closed successfully, order ID: {result.order}")

        return result

    def execute_trade(self, prediction):
        print("Executing trade...")
        lot = 0.1
        current_positions = mt5.positions_get(symbol=self.symbol)
        current_position_type = current_positions[0].type if current_positions else None

        if prediction == 1:
            price = mt5.symbol_info_tick(self.symbol).ask
            stop_loss = price - 0.01
            take_profit = price + 0.01
            if not current_positions or current_position_type == mt5.ORDER_TYPE_SELL:
                if current_position_type == mt5.ORDER_TYPE_SELL:
                    self.close_order(current_positions[0].ticket)
                result = self.place_order(mt5.ORDER_TYPE_BUY, lot, price, stop_loss, take_profit)
                print("Buy order result:", result)
        elif prediction == 0:
            price = mt5.symbol_info_tick(self.symbol).bid
            stop_loss = price + 0.01
            take_profit = price - 0.01
            if not current_positions or current_position_type == mt5.ORDER_TYPE_BUY:
                if current_position_type == mt5.ORDER_TYPE_BUY:
                    self.close_order(current_positions[0].ticket)
                result = self.place_order(mt5.ORDER_TYPE_SELL, lot, price, stop_loss, take_profit)
                print("Sell order result:", result)
        print("Trade executed")

    def trade(self):
        print("Starting trading loop...")
        self.running = True
        while self.running:
            print("Trading iteration started")
            data = self.fetch_data()
            if data.empty:
                print("No data fetched, skipping this iteration.")
                time.sleep(30)
                continue
            data = self.calculate_features(data)
            data = self.make_predictions(data)
            latest_prediction = data['Predicted_Signal'].iloc[-1]
            print("Latest prediction:", latest_prediction)
            self.execute_trade(latest_prediction)
            print("Sleeping for 1 hour")
            time.sleep(30)  # Sleep for 1 hour (adjust as needed)
            print("Woke up, starting next iteration")

    def start(self):
        print("Starting bot...")
        self.thread = threading.Thread(target=self.trade)
        self.thread.start()

    def stop(self):
        print("Stopping bot...")
        self.running = False
        self.thread.join()
        print("Bot stopped")

# Example usage:
if __name__ == "__main__":
    bot = SVMTradingBot(
        account=5030595537,
        password="-r7lRdVn",
        server="MetaQuotes-Demo",
        model_path="../models/svm_model.pkl"
    )
    bot.start()

    # Run for a certain period or based on a condition
    time.sleep(300)  # Run for 5 minutes
    bot.stop()


Initializing SVMTradingBot...
Connecting to MetaTrader 5...
Connected to MetaTrader 5
Starting bot...
Starting trading loop...
Trading iteration started
Fetching data...
Data fetched successfully. DataFrame shape: (10000, 8)
         time     open     high      low    close  tick_volume  spread  \
0  1683082800  1.10108  1.10206  1.10049  1.10173         1254       0   
1  1683086400  1.10174  1.10242  1.10129  1.10228         1523       0   
2  1683090000  1.10228  1.10258  1.10197  1.10253         1189       0   
3  1683093600  1.10253  1.10299  1.10251  1.10295          805       0   
4  1683097200  1.10295  1.10296  1.10214  1.10218         1156       0   

   real_volume  
0            0  
1            0  
2            0  
3            0  
4            0  
Calculating features...
Features calculated successfully
Making predictions...
Predictions made successfully
Latest prediction: 1.05711
Executing trade...
Trade executed
Sleeping for 1 hour
Woke up, starting next iteration
Tradi

In [3]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import joblib
from datetime import datetime, timedelta
import pytz

# Initialize and connect to MetaTrader 5
if not mt5.initialize():
    print("initialize() failed, error code =", mt5.last_error())
    quit()

account = 5030595537
password = "-r7lRdVn"
server = "MetaQuotes-Demo"

authorized = mt5.login(account, password=password, server=server)
if not authorized:
    print("Failed to connect to account. Error:", mt5.last_error())
    quit()

print("Connected to MetaTrader 5")

# Parameters
symbol = "EURUSD"
timeframe = mt5.TIMEFRAME_M1
model_path = "../models/svm_model.pkl"
day_to_backtest = "2023-12-01"  # Change this to the specific day you want to backtest

# Load the SVM model
model = joblib.load(model_path)

# Define the date range
start_date = datetime.strptime(day_to_backtest, "%Y-%m-%d")
end_date = start_date + timedelta(days=1)

# Convert to UTC
start_date = start_date.astimezone(pytz.utc)
end_date = end_date.astimezone(pytz.utc)

# Fetch data
print("Fetching data...")
rates = mt5.copy_rates_range(symbol, timeframe, start_date, end_date)
if rates is None:
    print("Failed to fetch data. Error:", mt5.last_error())
    quit()

data = pd.DataFrame(rates)
data['time'] = pd.to_datetime(data['time'], unit='s')
data.set_index('time', inplace=True)

# Feature calculation
def calculate_features(data):
    data['Returns'] = data['close'].pct_change()
    data['SMA_5'] = data['close'].rolling(window=5).mean()
    data['SMA_10'] = data['close'].rolling(window=10).mean()
    data['SMA_20'] = data['close'].rolling(window=20).mean()
    data['EMA_5'] = data['close'].ewm(span=5, adjust=False).mean()
    data['EMA_10'] = data['close'].ewm(span=10, adjust=False).mean()
    data['Momentum'] = data['close'] - data['close'].shift(5)
    data['Volatility'] = data['Returns'].rolling(window=5).std()
    data['ROC_5'] = data['close'].pct_change(periods=5)
    data['Price_Change'] = data['close'].pct_change()
    data['Lag_1'] = data['close'].shift(1)
    data['RSI'] = 100 - (100 / (1 + data['Returns'].rolling(window=14).mean() / abs(data['Returns'].rolling(window=14).mean())))
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    data.dropna(inplace=True)
    return data

data = calculate_features(data)

# Prepare the data for prediction
X = data[['SMA_5', 'SMA_10', 'SMA_20', 'EMA_5', 'EMA_10', 'Momentum', 'Volatility', 'ROC_5', 'Price_Change', 'Lag_1', 'RSI']]

# Fit and transform the scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Make predictions
data['Predicted_Signal'] = model.predict(X_scaled)

# Assuming you have actual signals to compare, you can replace the following with your actual signals
# For demonstration, let's create a random actual signal
np.random.seed(42)
data['Actual_Signal'] = np.random.choice([0, 1], size=len(data))

# Evaluate the model's performance
print("Model Performance:")
print(classification_report(data['Actual_Signal'], data['Predicted_Signal']))

# Disconnect from MetaTrader 5
mt5.shutdown()


Connected to MetaTrader 5
Fetching data...


ValueError: Found array with 0 sample(s) (shape=(0, 11)) while a minimum of 1 is required by StandardScaler.